In [1]:
from tinygrad import Tensor, UOp, dtypes, nn
from tinygrad.helpers import GlobalCounters, ContextVar, getenv
from tinygrad.nn.state import get_parameters

# LLM Memory in Tinygrad

> How much memory does your model use? Why does everyone say training an LLM takes `6*n_param*n_token` bytes? When does a KV cache slow down your model?

The goal of this blog is to turn you into a FLOPhead:

> **flophead** /ˈflopˌhɛd/ *noun* (informal): An AI researcher who specializes in optimizing model computational efficiency by reducing FLOPs (floating point operations) and improving hardware utilization.

We will analyze different versions of attention to determine the amount of memory and FLOPs each uses. You'll be able to answer questions like:
1. Why is my model slow?
2. How much memory does your model use?
3. Why does everyone say training an LLM takes `6*n_param*n_token` bytes?
4. When does a KV cache slow down your model?

# Vanilla Attention

Let's start with good old regular Grouped Query Attention:

In [17]:
class VanillaGQA:
    def __init__(self, dim:int, n_heads:int, n_kv_heads:int):
        self.n_heads      = n_heads
        self.n_kv_heads   = n_kv_heads
        self.head_dim     = dim // n_heads

        # --- attention projections (all linear, bias-free) ------------------
        kv_proj_out      = self.head_dim * n_kv_heads    # Llama-3 uses the same dim for K/V
        self.attn_q      = nn.Linear(dim, dim,         bias=False)
        self.attn_k      = nn.Linear(dim, kv_proj_out, bias=False)
        self.attn_v      = nn.Linear(dim, kv_proj_out, bias=False)
        self.attn_o = nn.Linear(dim, dim,         bias=False)

def __call__(self, x:Tensor, start_pos:int=0) -> Tensor:
    q, k, v = self.attn_q(x), self.attn_k(x), self.attn_v(x)

    B, T, _ = x.shape
    q = q.reshape(B, T, self.n_heads,    self.head_dim).transpose(1, 2)  # (B,H,T,Hd)
    k = k.reshape(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)  # (B,KvH,T,Hd)
    v = v.reshape(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)  # (B,KvH,T,Hd)

    # q = apply_rope(q, start_pos)
    # k = apply_rope(k, start_pos)

    mask = Tensor.full((1, 1, T, start_pos+T), float("-inf"), dtype=x.dtype, device=x.device).triu(start_pos+1) if T > 1 else None
    attn = q.scaled_dot_product_attention(k, v, attn_mask=mask, enable_gqa=True)     # (B,H,T,Hd)
    attn = attn.transpose(1, 2).reshape(B, T, -1)                                    # back to (B,T,D)
    return self.attn_o(attn)

How many FLOPS does doing a forward pass take?

* Computing $q$ takes $B S H D_H^2$ FLOPs 
* Computing $k, v$ takes $B S H_{k,v} D^2$ FLOPs 
* Computing $v$ takes $B S D^2$ FLOPs 

In tinygrad, it is very easy to see how much memory this model takes up. Let's initialize an attention model with a hidden dimension of `4` and `8` attention heads and `8` key value heads.

In [14]:
start_mem = GlobalCounters.mem_used
model = Attention(4, 8, 8)
for x in get_parameters(model): x.realize()
end_mem = GlobalCounters.mem_used

print(f'Model Weights = {end_mem - start_mem} bytes')

Model Weights = 128 bytes


Notice two things:
1. The model weights take up 140 bytes
2. Tinygrad is lazy. When we instantiate the model and call `Attention()`, we do not actually allocate space in memory for the model weights. Only when we call `realize()` on each parameter do we actually create space for the model in memory.

In [15]:
x = Tensor([1, 2, 3, 4, 5, 6, 7, 8])
model(x, 0)

TypeError: 'Attention' object is not callable

In [12]:
model.__call__()

AttributeError: 'Attention' object has no attribute '__call__'